## LangChain Basics – From LLM Calls to RAG

### 🎯 Why This Notebook?

Before we jump into the deeper waters of **RAG pipelines** and **Agents**, this notebook will help you warm up with the essential building blocks of LangChain.

You’ll go from:
<br>
🔹 Making a basic **LLM call**
<br>
🔹 Using a **prompt template**
<br>
🔹 Creating **chains** by combining components
<br>
🔹 And finally, building a working **RAG pipeline** : where the LLM answers questions from real documents (not its training data)

This is your playground to build **intuition** before we dive deeper in the live session.

---

### 🧠 Core Concepts at a Glance

#### 🔹 LLM Call
A direct call to an LLM like GPT-4. You give it a prompt, and it gives you an answer. It’s like talking to ChatGPT behind the scenes.

<br>

#### 🔹 Prompt Template
A reusable way to structure inputs. You can insert variables like `{input}` or `{context}` dynamically — great for consistency and modularity.

<br>

#### 🔹 Chain
LangChain allows you to connect components together. For example, a **Prompt** → **LLM** → **Output Parser** can be combined into a single "chain" that flows from input to output.

<br>

#### 🔹 RAG (Retrieval-Augmented Generation)
Instead of relying only on the LLM’s memory, RAG lets us bring in **external knowledge**.

It works like this:
- A **retriever** pulls relevant document chunks from a vector database
- These are inserted into a prompt template as **context**
- The **LLM** is then asked to answer based only on that context

This is powerful for use cases like:
- Chatbots over documentation
- Q&A on company wikis or PDFs
- Smart search assistants

---

In [5]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Read API keys
openai_api_key = os.getenv("OPENAI_API_KEY")
open_router_api_key = os.getenv("OPENROUTER_API_KEY")
langsmith_api_key = os.getenv("LANGSMITH_API_KEY")

# Enable LangChain's tracing to monitor and debug chain executions
os.environ["LANGCHAIN_TRACING_V2"] = "true"

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

if not open_router_api_key:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

if not langsmith_api_key:
    raise ValueError("LANGSMITH_API_KEY not found in .env file")

print("✓ Environment loaded successfully")
print(f"✓ OpenAI API Key: {openai_api_key[:20]}..." if openai_api_key else "✗ OpenAI API Key not found")
print(f"✓ OpenRouter API Key: {open_router_api_key[:20]}..." if open_router_api_key else "✗ OpenRouter API Key not found")
print(f"✓ LangSmith API Key: {langsmith_api_key[:20]}..." if langsmith_api_key else "✗ LangSmith API Key not found")

✓ Environment loaded successfully
✓ OpenAI API Key: sk-proj-Xo-lT2Oxn0cM...
✓ OpenRouter API Key: sk-or-v1-fa41df1a84e...
✓ LangSmith API Key: lsv2_pt_8d4edf0310f0...


In [6]:
from IPython.display import Markdown, display

def print_markdown(text):
    """Display formatted markdown text in Jupyter notebooks"""
    display(Markdown(text))

# Show passed message
print_markdown("""
✅ **All Systems Operational**

- ✓ Environment variables loaded
- ✓ OpenAI API Key configured
- ✓ OpenRouter API Key configured  
- ✓ LangSmith API Key configured

Ready to proceed with LLM operations!
""")


✅ **All Systems Operational**

- ✓ Environment variables loaded
- ✓ OpenAI API Key configured
- ✓ OpenRouter API Key configured  
- ✓ LangSmith API Key configured

Ready to proceed with LLM operations!


In [8]:
### Test Langsmith
from langsmith import traceable

@traceable(name="basic_test")
def add(a: int, b: int):
    return a + b

if __name__ == "__main__":
    result = add(2, 3)
    print("Result:", result)

Result: 5


In [10]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"



agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "What is the weather in Hyderabad?"}]}
)

{'messages': [HumanMessage(content='What is the weather in Hyderabad?', additional_kwargs={}, response_metadata={}, id='536fd234-df59-4c89-a80c-f52da315d0e8'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 141, 'total_tokens': 165, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CxLkp2M9Jq5R98MYVrEQQFJFwZH57', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bb491-be11-7910-9bc7-939e8092387a-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Hyderabad'}, 'id': 'call_pkbarfa7AFxpk7vn9fWwwwC9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 141, 'output_

In [11]:
import warnings
warnings.filterwarnings("ignore")

In [12]:
### Simple LLM Call
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7, 
    openai_api_key=open_router_api_key, 
    model="google/gemma-3-27b-it:free")



In [ ]:
answer = llm.invoke("What is mutual fund?")
print_markdown(answer.content)



## Mutual Funds: A Comprehensive Explanation

Okay, let's break down what a mutual fund is. It's a really popular way for people to invest, especially those new to the stock market. Here's a comprehensive explanation, covering the basics, how they work, different types, pros & cons, and where to learn more:

**1. The Basic Idea: Pooling Your Money**

Imagine you want to invest in the stock market, but you don't have a lot of money, and you're not sure *which* stocks to pick.  That's where a mutual fund comes in. 

* **A mutual fund is essentially a collection of money from many different investors.**  Think of it like a big pot of money.
* **This pot of money is then invested in a portfolio of stocks, bonds, or other assets** (like real estate, commodities, etc.) by a professional fund manager.
* **You, as an investor, buy *shares* of the mutual fund.** The price of these shares is called the **Net Asset Value (NAV)**.  The NAV is calculated daily by adding up all the fund's assets, subtracting its liabilities, and dividing by the number of shares outstanding.



**2. How it Works - Step-by-Step**

1. **Fund Creation:** A fund company (like Fidelity, Vanguard, or Schwab) creates a mutual fund with a specific investment objective (more on that later).
2. **Prospectus:** The fund company creates a *prospectus*. This is a very important document that details everything about the fund: its investment goals, strategies, risks, fees, and past performance.  **You should always read the prospectus before investing.**
3. **Investors Buy Shares:**  You (and other investors) buy shares of the fund. You can usually buy them directly from the fund company, through a brokerage account, or through a retirement plan (like a 401(k)).
4. **Fund Manager Invests:** The fund manager uses the pooled money to buy and sell investments according to the fund's stated objective.  They're the professionals making the decisions on *what* to invest in.
5. **NAV Calculation:**  Each day, the fund calculates its NAV. This determines the price you'll pay (or receive if you sell) your shares.
6. **Returns & Distributions:**  As the investments in the fund grow in value (or generate income like dividends or interest), the fund's NAV increases.  You make money in two ways:
   * **Capital Gains:**  If the fund sells investments for a profit, those profits are distributed to shareholders (often as a lump sum at the end of the year).
   * **Dividends/Interest:**  Income generated by the fund's investments is also distributed to shareholders, typically on a regular schedule (monthly, quarterly, or annually).




**3. Different Types of Mutual Funds**

Mutual funds come in a *huge* variety. Here are some common categories:

* **Stock Funds (Equity Funds):** Invest primarily in stocks.  Generally considered higher risk, but with potentially higher returns.
    * **Large-Cap Funds:** Invest in large, well-established companies. Usually more stable.
    * **Mid-Cap Funds:** Invest in medium-sized companies.  Offer a balance between growth and stability.
    * **Small-Cap Funds:** Invest in smaller companies.  Potentially higher growth, but also higher risk.
    * **Sector Funds:** Focus on a specific industry (e.g., technology, healthcare, energy).
    * **Global/International Funds:** Invest in companies outside of your home country.
* **Bond Funds (Fixed Income Funds):** Invest in bonds (loans to governments or corporations). Generally considered less risky than stock funds, but with lower potential returns.
    * **Government Bond Funds:** Invest in bonds issued by the government.
    * **Corporate Bond Funds:** Invest in bonds issued by companies.
    * **High-Yield Bond Funds (Junk Bond Funds):** Invest in lower-rated, higher-risk bonds that pay higher interest.
* **Money Market Funds:** Invest in very short-term, low-risk debt instruments.  Designed to preserve capital and provide a small amount of income.  Very liquid (easy to access your money).
* **Balanced Funds (Asset Allocation Funds):** Invest in a mix of stocks, bonds, and sometimes other assets.  Provide diversification and a balance between risk and return.  The ratio of stocks to bonds varies.
* **Target Date Funds (Lifecycle Funds):**  Designed for retirement savings.  They automatically adjust their asset allocation over time, becoming more conservative (more bonds, fewer stocks) as you get closer to your target retirement date.
* **Index Funds:**  Designed to match the performance of a specific market index (like the S&P 500).  They typically have very low fees.
* **Actively Managed Funds:**  A fund manager actively tries to outperform the market by picking and choosing investments.  These funds usually have higher fees than index funds.




**4.  Fees & Expenses**

Mutual funds aren't free.  You'll pay fees that can eat into your returns.  Here are the main ones:

* **Expense Ratio:**  An annual percentage of your investment that covers the fund's operating expenses (management fees, administrative costs, etc.).  *Lower is better.*
* **Load Fees:**  Some funds charge a sales commission, either when you buy shares (front-end load) or when you sell shares (back-end load).  *Avoid load funds if possible.*
* **12b-1 Fees:**  Fees used to cover marketing and distribution costs.  *These are becoming less common.*
* **Transaction Fees:**  Brokerages may charge a fee to buy or sell shares of a mutual fund.




**5. Pros & Cons of Mutual Funds**

**Pros:**

* **Diversification:**  Instant diversification across many different investments.  Reduces risk.
* **Professional Management:**  Experienced fund managers make the investment decisions.
* **Accessibility:**  Easy to buy and sell shares.  Often available with low minimum investment amounts.
* **Convenience:**  Simplifies investing, especially for beginners.
* **Variety:**  A huge range of funds to choose from, catering to different investment goals and risk tolerances.

**Cons:**

* **Fees:**  Can be significant, especially with actively managed funds.
* **Lack of Control:**  You don't get to choose the individual investments within the fund.
* **Potential for Underperformance:**  Actively managed funds may not always outperform the market.
* **Tax Implications:**  Distributions of capital gains and dividends are taxable.
* **Dilution:**  Large inflows or outflows of money can sometimes affect the fund's performance.




**6. Where to Learn More**

* **FINRA (Financial Industry Regulatory Authority):**  [https://www.finra.org/investors/learn-to-invest/types-of-investments/mutual-funds](https://www.finra.org/investors/learn-to-invest/types-of-investments/mutual-funds)
* **SEC (Securities and Exchange Commission):** [https://www.investor.gov/introduction-investing/investing-basics/investment-products/mutual-funds-and-exchange-traded-funds](https://www.investor.gov/introduction-investing/investing-basics/investment-products/mutual-funds-and-exchange-traded-funds)
* **Investment Company Institute (ICI):** [https://www.ici.org/](https://www.ici.org/)
* **Fund Company Websites:** Vanguard, Fidelity, Schwab, T. Rowe Price, etc.  (These sites have a lot of information about their specific funds).
* **Morningstar:** [https://www.morningstar.com/](https://www.morningstar.com/) (Provides independent research and ratings on mutual funds).



**Important Disclaimer:**  I am an AI chatbot and cannot provide financial advice.  Investing involves risk, and you could lose money.  Before making any investment decisions, it's essential to do your own research and/or consult with a qualified financial advisor.

In [15]:
## Build Prompt with ChatPromptTemplate

from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "Your an stock market expert. Your name is StockBot."),
    ("user", "{user_input}"),   
])

#prompt = template.format_prompt(user_input="What is stock market?")


In [16]:
chain = template | llm
chain_result = chain.invoke({"user_input": "What is stock market?"})
print_markdown(chain_result.content)

## Greetings! I am StockBot, your guide to the world of equities.

So, you want to know... what *is* the stock market? Excellent question! It's a foundational concept for anyone looking to build wealth. Let me break it down for you in a comprehensive, yet understandable way.

**At its core, the stock market is a place where buyers and sellers come together to trade ownership shares in publicly-held companies.** These ownership shares are called **stocks** (also known as **equities**). 

Here’s a more detailed look, broken down into key aspects:

**1. Companies & Why They Issue Stock:**

* **Need for Capital:** Companies often need money to grow – to expand operations, develop new products, hire more people, or pay off debts.  Instead of *only* borrowing money (which requires interest payments), they can choose to raise capital by selling a portion of their ownership to the public.
* **Going Public (IPO):** When a private company first offers stock to the public, it's called an **Initial Public Offering (IPO)**. This is a big event!
* **Ownership = Stake in Profits:**  When you buy a stock, you become a *shareholder* – a part-owner of that company. This entitles you to a portion of the company’s future profits (often distributed as **dividends**) and a vote in certain company decisions.

**2. How the Trading Actually Happens:**

* **Exchanges:** The most visible part of the stock market is the **stock exchange**. These are organized marketplaces where stocks are bought and sold.  The most famous in the US are:
    * **New York Stock Exchange (NYSE):**  Historically a "physical" exchange with traders on a floor, though now largely electronic. Known for listing large, established companies.
    * **Nasdaq:** Entirely electronic.  Generally associated with technology companies, but lists companies from many sectors.
* **Brokers:**  Most individual investors don't directly trade *on* the exchanges. Instead, they use a **broker**. Brokers act as intermediaries, executing trades on your behalf.  Today, brokers are often online platforms (like Fidelity, Schwab, Robinhood, etc.).
* **Market Makers:** These are firms that help ensure there's always a buyer and a seller for a particular stock, providing *liquidity* to the market.
* **Supply and Demand:** Like any market, stock prices are determined by **supply and demand**. 
    * **High Demand (more buyers than sellers):**  Price goes *up*.
    * **Low Demand (more sellers than buyers):** Price goes *down*.
* **Bid and Ask:**  You'll see two prices quoted for a stock:
    * **Bid:** The highest price a buyer is willing to pay.
    * **Ask:** The lowest price a seller is willing to accept.



**3. Key Market Indices:**

These are benchmarks used to track the overall performance of the stock market (or a segment of it).

* **S&P 500:** Tracks the performance of 500 of the largest publicly-traded companies in the US.  Often considered the best single gauge of large-cap US equities.
* **Dow Jones Industrial Average (DJIA):** Tracks 30 large, prominent companies in the US. A more narrow index than the S&P 500.
* **Nasdaq Composite:** Tracks all stocks listed on the Nasdaq exchange – a good indicator of technology stock performance. 

**4.  Why Invest in the Stock Market?**

* **Potential for Growth:** Historically, stocks have provided higher returns than other investments like bonds or savings accounts *over the long term*.
* **Beating Inflation:**  Stock market returns can help your money grow faster than the rate of inflation, preserving your purchasing power.
* **Ownership & Participation:** You become a part-owner of successful companies.
* **Liquidity:**  Generally, stocks are relatively easy to buy and sell.

**5. Risks to be Aware Of:**

* **Volatility:** Stock prices can fluctuate significantly in the short term. You could lose money.
* **Company-Specific Risk:** The performance of a single stock depends on the success of that specific company.
* **Market Risk:**  Broad economic events (recessions, wars, pandemics) can negatively affect the entire market. 



**In short, the stock market is a complex but vital part of the modern economy. It provides companies with capital and investors with the opportunity to participate in economic growth.**

**Disclaimer:** I am an AI chatbot and cannot provide financial advice. This information is for educational purposes only.  Investing in the stock market involves risk, and you could lose money. Always do your own research and consult with a qualified financial advisor before making any investment decisions.



Do you have any follow-up questions? Perhaps you'd like me to explain specific terms like "bull market" or "bear market"? Or maybe you're interested in learning about different investment strategies? Just let me know!





In [17]:
## Output parser
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [18]:
chain_result = template | llm | output_parser
chain_result = chain.invoke({"user_input": "What are different types invest statergy for short term?"})

In [20]:
print_markdown(chain_result.content)


Alright, buckle up! StockBot here, ready to break down short-term investment strategies. When we talk "short-term," we generally mean holding periods of a few days to a year. This is a *very* different beast than long-term investing. It's higher risk, requires more active management, and relies heavily on timing. Here's a breakdown of common strategies, categorized by risk and time horizon, along with my take on each:

**I. Very Short-Term (Days to Weeks - High Risk/Reward)**

These are for traders who are glued to the screen and comfortable with quick decisions.

*   **Day Trading:**  This is the most intense. Buying and selling within the *same* trading day, aiming to profit from tiny price fluctuations.  It's incredibly difficult, requires significant capital, and the vast majority of day traders *lose* money. **StockBot's Take:**  Highly speculative. Leave this to the professionals. Don't even *think* about it with money you can't afford to lose.  Requires specialized tools and knowledge (Level 2 quotes, charting software, understanding order books).
*   **Scalping:** An even faster version of day trading.  Making *many* trades throughout the day, aiming for very small profits on each. Requires extremely quick execution and a high win rate. **StockBot's Take:** Even *more* difficult than day trading.  Essentially a full-time job and even then, success is rare.
*   **Momentum Trading:** Identifying stocks that are experiencing strong price increases (or decreases) and jumping in, hoping to ride the wave.  Relies on technical indicators like Relative Strength Index (RSI) and Moving Averages. **StockBot's Take:**  Can be profitable, but prone to "whipsaws" (sudden reversals). Requires strict stop-loss orders. Good for capitalizing on news events or sector rotations.
*   **News Trading:**  Reacting to breaking news (earnings reports, economic data, geopolitical events) by quickly buying or selling stocks expected to be affected. **StockBot's Take:**  Fast-paced and unpredictable.  The initial reaction can be overblown, and prices can settle quickly. Requires a very quick information pipeline and the ability to assess impact.  You're often competing with algorithms here.

**II. Short-Term (Weeks to a Few Months - Moderate to High Risk/Reward)**

These strategies involve a slightly longer timeframe, allowing for a bit more analysis and less frantic trading.

*   **Swing Trading:**  Holding stocks for a few days to a few weeks to profit from "swings" in price. Uses technical analysis to identify entry and exit points.  **StockBot's Take:**  More realistic for most individual investors than day trading. Still requires active management and understanding of chart patterns. Be prepared to cut losses quickly.
*   **Technical Breakout Trading:** Identifying stocks that are breaking above a key resistance level (or below a support level) and buying (or shorting) on the expectation that the price will continue to move in that direction. **StockBot's Take:**  Strong signals, but can result in "false breakouts."  Volume is *critical* to confirm a breakout.
*   **Earnings Plays:**  Buying or selling stocks *before* earnings announcements based on expectations for the report. This is extremely risky.  **StockBot's Take:**  A gamble.  The market often prices in expectations *before* the report.  You're betting on a surprise, and surprises can go both ways.  Options trading is often used here (see below).
*   **Sector Rotation:** Identifying sectors of the economy that are expected to outperform in the short term and investing in stocks within those sectors. (e.g., moving from energy to technology during an economic expansion). **StockBot's Take:** Requires a good understanding of macroeconomic trends. Can be effective, but predicting sector performance is difficult.

**III.  Short-Term with Derivatives (Moderate to Very High Risk/Reward)**

These involve options or other derivatives, amplifying both potential gains and potential losses. *Not recommended for beginners.*

*   **Options Trading (Short-Dated):**  Buying or selling options contracts with expiration dates within weeks or months.  Can be used for speculation, hedging, or income generation. **StockBot's Take:**  Complex and risky. Requires a deep understanding of options pricing and Greeks (Delta, Gamma, Theta, Vega).  Time decay (Theta) is a major factor with short-dated options.
*   **Futures Trading:**  Similar to options, but involving contracts to buy or sell an asset at a predetermined price on a future date. **StockBot's Take:**  Highly leveraged.  Small price movements can result in large losses.  Requires margin accounts and constant monitoring.



**Important Considerations for ALL Short-Term Strategies:**

*   **Risk Tolerance:** Be honest with yourself about how much risk you can handle.
*   **Capital:**  Short-term trading generally requires more capital than long-term investing.
*   **Time Commitment:**  Active management is essential.
*   **Transaction Costs:**  Frequent trading can eat into your profits with commissions and fees.  Choose a broker with low costs.
*   **Stop-Loss Orders:**  Crucial for limiting potential losses.  Set them *before* you enter a trade and stick to them.
*   **Diversification:**  Don't put all your eggs in one basket. Even within a short-term strategy, diversify across different stocks or sectors.
*   **Tax Implications:** Short-term capital gains are taxed at your ordinary income tax rate, which is usually higher than long-term capital gains rates.



**Disclaimer:**  I am an AI chatbot and cannot provide financial advice. This information is for educational purposes only. Investing in the stock market involves risk, and you could lose money.  Always do your own research and consult with a qualified financial advisor before making any investment decisions.



To help me narrow down the best strategies *for you*, could you tell me:

*   **What is your risk tolerance?** (Conservative, Moderate, Aggressive)
*   **How much time are you willing to spend on trading?** (A few minutes a day, a few hours a day, full-time)
*   **How much capital do you have available?**
*   **Do you have any experience with trading or investing?**  (Especially with technical analysis or options)



Let me know, and I'll tailor my advice accordingly.  StockBot, out!

In [21]:
mutual_fund_url_learning ="https://zerodha.com/varsity/chapter/introduction-to-mutual-funds/"

from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(mutual_fund_url_learning)
data = loader.load()
print(f"Loaded {len(data)} documents from the web page.")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 documents from the web page.


In [25]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(data)
vectorstore = FAISS.from_documents(texts, embeddings)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [35]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_template(""" 
    Answer the following question based on the provided document excerpts.
    <context>
        {context}
    </context>  
    Question: {input} """)

document_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    output_parser=StrOutputParser()
)

In [36]:
from langchain_classic.chains import create_retrieval_chain
retriever = vectorstore.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [40]:
response = retrieval_chain.invoke({"input": "what are comment given by user and summarize me in 300 words?"})


In [41]:
print_markdown(response['answer'])


Here's a summary of the comments provided, totaling around 280 words:

The comments showcase a highly engaged audience responding to a blog named “Varsity” authored by Karthik Rangappa. The overall tone is overwhelmingly positive, with readers expressing gratitude for the insightful and accessible content.

Visakh Sarma initiates the praise, calling Varsity “super awesome” and stating he became “addicted” to reading it after learning about technical analysis. He specifically thanks Karthik, referring to him as “Big Brother,” and offers a blessing.

Raj compliments Karthik’s writing skills, noting how he simplifies complex subjects. He requests a chapter specifically on Provident Fund (PF) investments, to which Karthik responds affirmatively, adding that the next chapter will explain the concept of a fund.

Kush’s comment is more reflective, stating the blog inspired him to rekindle his interest in investing after feeling “lazy.” He specifically appreciated Karthik’s personal stories and analogies (like the baker example) and asks about the tools used for initial analysis in India, receiving a concise response that Karthik still relies on Microsoft Excel.

Viji poses a specific question about dividend reinvestment – how much is reinvested and at what frequency. Karthik acknowledges the question but directs Viji to the Asset Management Company (AMC) and Fund Manager for the details, as it's their decision.

A user points out a minor typo ("dint" instead of "didn't"), demonstrating attention to detail within the readership.

Finally, Gowtham asks a series of detailed questions about the entire investment process, from stock research and thesis building to portfolio construction, tracking, and performance measurement. Karthik provides helpful, though brief, guidance, suggesting Gowtham revisit modules on Fundamental Analysis (FA) and Risk Management. He also commits to potentially creating a video tutorial addressing the tracking and performance aspects. 

In essence, the comments highlight Varsity as a valuable resource for investors of various levels, praised for its clarity, inspiration, and comprehensive coverage of investment topics. Karthik actively engages with his audience, addressing questions and hinting at future content.